In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install transformers peft datasets accelerate bitsandbytes trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 31.2 MB/s eta 0:00:00


In [4]:
import os
import re
import json
import torch
import numpy as np
from tqdm import tqdm

In [5]:
#config
MODEL_NAME = 'facebook/opt-125m'
OUTPUT_DIR = './lora-output'
ADAPTER_DIR = './lora-adapter'
BATCH_SIZE = 16

USE_4BIT = torch.cuda.is_available()
TRAIN_SAMPLE = 2080
EVAL_SAMPLE = 200
MAX_LENGTH = 256

In [6]:
from datasets import load_dataset
raw = load_dataset('databricks/databricks-dolly-15k', split='train')
len(raw), raw.column_names

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

(15011, ['instruction', 'context', 'response', 'category'])

In [7]:
def format_example(example):
    if example['context'] and example['context'].strip():
        text = (
            f"### Instruction:\n{example['instruction'].strip()}"
            f"### Context:\n{example['context'].strip()}"
            f"### Response:\n{example['response'].strip()}"
        )
    else:
         text = (
            f"### Instruction:\n{example['instruction'].strip()}"
            f"### Response:\n{example['response'].strip()}"
        )

    return {'text': text}

In [8]:
dataset = raw.map(format_example, remove_columns=raw.column_names)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [9]:
if TRAIN_SAMPLE:
    total = min(TRAIN_SAMPLE + EVAL_SAMPLE, len(dataset))
    dataset = dataset.select(range(total))

split = dataset.train_test_split(test_size=.1, seed=42)
train_data = split['train']
eval_data = split['test']

In [10]:
dataset[0]

{'text': "### Instruction:\nWhen did Virgin Australia start operating?### Context:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.### Response:\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route."}

In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [12]:
def tokenize(example):
    result = tokenizer(
        example['text'],  
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result  

In [13]:
tokenized_train = train_data.map(tokenize, batched=True, remove_columns=['text'])
tokenized_eval = eval_data.map(tokenize, batched=True, remove_columns=['text'])

tokenized_train.set_format('torch') 
tokenized_eval.set_format('torch')

Map:   0%|          | 0/2052 [00:00<?, ? examples/s]

Map:   0%|          | 0/228 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoModelForCausalLM,BitsAndBytesConfig
from peft import (
LoraConfig,
get_peft_model,
TaskType,
prepare_model_for_kbit_training
)

In [15]:
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = 'nf4',
        bnb_4bit_compute_dtype = torch.float16,
        bnb_4bit_use_double_quant = True
        
    )
    model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map = 'auto'
    )

    model = prepare_model_for_kbit_training(model)
    print("loaded in 4-bit Qlora mode")

else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype = torch.float32,
        device_map = 'auto'
    )

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

loaded in 4-bit Qlora mode


In [16]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"{name:<50} {module}")

model.decoder.layers.0.self_attn.k_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.0.self_attn.v_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.0.self_attn.q_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.0.self_attn.out_proj          Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.0.fc1                         Linear4bit(in_features=768, out_features=3072, bias=True)
model.decoder.layers.0.fc2                         Linear4bit(in_features=3072, out_features=768, bias=True)
model.decoder.layers.1.self_attn.k_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.1.self_attn.v_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.1.self_attn.q_proj            Linear4bit(in_features=768, out_features=768, bias=True)
model.decoder.layers.1.sel

In [17]:
TARGET_MODULES = ['q_proj', 'v_proj']
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.1,
    bias = 'none',
    target_modules = TARGET_MODULES
    
)

model = get_peft_model(model, lora_config)

In [18]:
model.print_trainable_parameters()

trainable params: 294,912 || all params: 164,143,104 || trainable%: 0.1797


In [19]:
from transformers import (
TrainingArguments, 
Trainer, 
DataCollatorForLanguageModeling,
EarlyStoppingCallback
)

In [20]:
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps= 4,
    learning_rate = 0.002,
    fp16 = torch.cuda.is_available(),
    logging_steps = 10,
    eval_strategy = 'steps',
    eval_steps = 50,
    save_strategy = 'steps',
    load_best_model_at_end = True,
    metric_for_best_model = 'eval_loss',
    greater_is_better = False,
    optim = 'paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
    gradient_checkpointing = USE_4BIT,
    warmup_ratio = .05,
    lr_scheduler_type =  'cosine',
    dataloader_pin_memory = False

)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [22]:
trainer = Trainer(
    model= model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_eval,
    data_collator= data_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
    
)

In [23]:
train_result = trainer.train()

Step,Training Loss,Validation Loss
50,2.893535,2.688100
100,2.790168,2.644655
150,2.707167,2.630398
200,2.737786,2.626655
250,2.683556,2.618499
300,2.647705,2.610460
350,2.662622,2.614805


In [24]:
model = model.merge_and_unload()  # merges LoRA weights into base model

model.save_pretrained("./saved_model")



# Save fine-tuned weights + tokenizer

trainer.save_model("./saved_model")

tokenizer.save_pretrained("./saved_model")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_model/tokenizer_config.json', './saved_model/tokenizer.json')

In [25]:
results = trainer.evaluate()
print(results)  # gives eval_loss, perplexity, etc.

# Perplexity (standard metric for causal LMs)
import math
perplexity = math.exp(results['eval_loss'])
print(f"Perplexity: {perplexity:.2f}")

{'eval_loss': 2.6302998065948486, 'eval_runtime': 3.0388, 'eval_samples_per_second': 75.03, 'eval_steps_per_second': 18.758, 'epoch': 3.0}
Perplexity: 13.88


In [26]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load
model = AutoModelForCausalLM.from_pretrained("./saved_model")
tokenizer = AutoTokenizer.from_pretrained("./saved_model")
model.eval()

def generate(prompt, max_new_tokens=200, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (strip the prompt)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(lines=4, placeholder="Enter your prompt...", label="Input"),
        gr.Slider(50, 500, value=200, label="Max new tokens"),
        gr.Slider(0.1, 1.5, value=0.7, label="Temperature"),
    ],
    outputs=gr.Textbox(label="Response"),
    title="My Fine-tuned LLM",
)

demo.launch()  # add share=True to get a public link

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights: 0it [00:00, ?it/s]

OPTForCausalLM LOAD REPORT from: ./saved_model
Key                                                                  | Status  | 
---------------------------------------------------------------------+---------+-
model.decoder.layers.{0...11}.self_attn.v_proj.lora_A.default.weight | MISSING | 
model.decoder.layers.{0...11}.self_attn.v_proj.lora_B.default.weight | MISSING | 
model.decoder.layers.{0...11}.self_attn.q_proj.lora_B.default.weight | MISSING | 
model.decoder.layers.{0...11}.self_attn.q_proj.lora_A.default.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://f7cb0a1e00764601d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
